In [2]:
# NB11 Cell 1 — Project setup + artifact inventory
# Local-first. Same project-root discovery as NB09.
# Read-only for source artifacts. Writes only nb11_artifacts/cell1_artifact_inventory.csv.

from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# ---- Project root discovery (same pattern as NB09) ----
CURRENT_DIR = Path.cwd().resolve()

def discover_project_root():
    # Primary anchor: both nb08_artifacts and data/raw under the same parent
    for p in (CURRENT_DIR, *CURRENT_DIR.parents):
        if (p / "nb08_artifacts").is_dir() and (p / "data" / "raw").is_dir():
            return p
    # Fallback: nb08_artifacts alone (NB11 does not need data/raw unless re-scoring)
    for p in (CURRENT_DIR, *CURRENT_DIR.parents):
        if (p / "nb08_artifacts").is_dir():
            return p
    return None

PROJECT_ROOT = discover_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root.\n"
        "Expected a parent folder containing 'nb08_artifacts/'.\n"
        f"Current working directory: {CURRENT_DIR}\n"
        "Either cd into the project folder or open the notebook from there."
    )

# ---- Standard paths (same pattern as NB09) ----
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
NB08_DIR = PROJECT_ROOT / "nb08_artifacts"
NB09_DIR = PROJECT_ROOT / "nb09_artifacts"

# NB10 may be extracted flat or nested; check both.
NB10_CANDIDATES = [
    PROJECT_ROOT / "nb10_artifacts",
    PROJECT_ROOT / "nb10_artifacts" / "nb10_artifacts",
]
NB10_DIR = next((p for p in NB10_CANDIDATES if p.is_dir()), None)

NB11_OUT_DIR = PROJECT_ROOT / "nb11_artifacts"
NB11_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Fixed method lists (matching NB09) ----
METHODS = ["rule", "rolling", "simple", "if", "ocsvm", "lof", "lstm", "tcn", "usad"]
SCORE_METHODS = ["tcn", "lstm", "usad", "if", "ocsvm", "lof"]
BASELINE_METHODS = ["rule", "rolling", "simple"]

# ---- Required artifacts ----
REQUIRED = {
    "NB08": {
        "root": NB08_DIR,
        "items": [
            "evaluation_summary.csv",
            "final_sequence_scores.npz",
            "final_observation_scores.npz",
            "evaluation_sequences.npz",
            "attack_evaluation_sequences.npz",
            "frozen_thresholds.json",
            "scaler.joblib",
        ],
    },
    "NB09": {
        "root": NB09_DIR,
        "items": [
            "event_recall.csv",
            "event_vs_point.csv",
            "detection_delays.csv",
            "detection_delay_summary.csv",
            "scenario_detection.csv",
            "scenario_detection_summary.csv",
            "fp_summary.csv",
            "fn_summary.csv",
            "validation_false_alarm_summary.csv",
            "tcn_lstm_spearman.csv",
            "simple_tcn_lstm_scenario_comparison.csv",
        ],
    },
    "NB10": {
        "root": NB10_DIR,
        "items": [
            "batch_scoring/window_sensitivity_summary.csv",
            "batch_scoring/scaling_sensitivity_summary.csv",
            "batch_scoring/seed_stability_summary.csv",
            "batch_scoring/threshold_sensitivity_summary.csv",
            "batch_scoring/threshold_invariant_reference.csv",
            "synthesis/nb10_cross_axis_summary.csv",
            "synthesis/nb10_axis_rollup.csv",
            "synthesis/nb10_threshold_swing.csv",
            "synthesis/NB10_SYNTHESIS_REPORT.md",
            "window_scores/window_scores.npz",
            "window_scores/window_scores_metadata.json",
        ],
    },
}

# ---- Build inventory ----
rows = []
for group, spec in REQUIRED.items():
    root = spec["root"]
    for item in spec["items"]:
        p = (root / item) if root else None
        present = bool(p and p.is_file())
        rows.append({
            "group": group,
            "item": item,
            "present": present,
            "size_bytes": p.stat().st_size if present else None,
            "resolved_path": str(p) if present else "",
        })

df = pd.DataFrame(rows)

# ---- Print summary ----
# print(f"Project root: {PROJECT_ROOT}")
# print(f"NB08 dir:     {NB08_DIR}  ({'FOUND' if NB08_DIR.is_dir() else 'MISSING'})")
# print(f"NB09 dir:     {NB09_DIR}  ({'FOUND' if NB09_DIR.is_dir() else 'MISSING'})")
# print(f"NB10 dir:     {NB10_DIR if NB10_DIR else 'MISSING'}")
# print(f"NB11 output:  {NB11_OUT_DIR}")
print()

for group in ("NB08", "NB09", "NB10"):
    sub = df[df["group"] == group]
    ok = int(sub["present"].sum())
    print(f"[{group}] {ok}/{len(sub)} present")
    for _, r in sub.iterrows():
        mark = "✓" if r["present"] else "✗"
        size = f"{r['size_bytes']:>12,}" if r["size_bytes"] else " " * 12
        print(f"  {mark} {r['item']:58s} {size}")
    print()

# ---- Verdict ----
missing_by_group = {
    g: df[(df["group"] == g) & (~df["present"])]["item"].tolist()
    for g in ("NB08", "NB09", "NB10")
}

critical_missing = missing_by_group["NB08"] + missing_by_group["NB09"]
nb10_missing = missing_by_group["NB10"]
only_window_missing = nb10_missing == [
    "window_scores/window_scores.npz",
    "window_scores/window_scores_metadata.json",
]

print("=" * 72)
if not critical_missing and not nb10_missing:
    print("All required artifacts present. Proceed to Cell 2.")
elif not critical_missing and only_window_missing:
    print("All critical artifacts present.")
    print("window_scores missing — Cell 3 will need to re-score the 4 window models.")
    print("This requires normal.csv + attack.csv + PyTorch (CPU).")
    print("Proceed to Cell 2. Cell 3 will branch on this case.")
elif critical_missing:
    print("CRITICAL missing artifacts:")
    for item in critical_missing:
        print(f"  - {item}")
    print("Fix these before proceeding.")
elif nb10_missing:
    print("NB10 artifacts incomplete:")
    for item in nb10_missing:
        print(f"  - {item}")
    print("Re-extract nb10_artifacts.zip if the folder exists but files are missing.")
print("=" * 72)

# ---- Save inventory ----
inv_path = NB11_OUT_DIR / "cell1_artifact_inventory.csv"
df.to_csv(inv_path, index=False)
# print(f"\nInventory saved: {inv_path}")

# ---- Handles for later cells ----
EVALUATION_SUMMARY = NB08_DIR / "evaluation_summary.csv"
FINAL_SEQUENCE_SCORES = NB08_DIR / "final_sequence_scores.npz"
FINAL_OBSERVATION_SCORES = NB08_DIR / "final_observation_scores.npz"
FROZEN_THRESHOLDS = NB08_DIR / "frozen_thresholds.json"
EVALUATION_SEQUENCES = NB08_DIR / "evaluation_sequences.npz"
ATTACK_EVALUATION_SEQUENCES = NB08_DIR / "attack_evaluation_sequences.npz"

WINDOW_SCORES = (NB10_DIR / "window_scores" / "window_scores.npz") if NB10_DIR else None
WINDOW_SCORES_META = (NB10_DIR / "window_scores" / "window_scores_metadata.json") if NB10_DIR else None

NORMAL_CSV = DATA_RAW_DIR / "normal.csv"
ATTACK_CSV = DATA_RAW_DIR / "attack.csv"


[NB08] 7/7 present
  ✓ evaluation_summary.csv                                          1,872.0
  ✓ final_sequence_scores.npz                                   5,063,919.0
  ✓ final_observation_scores.npz                                2,927,982.0
  ✓ evaluation_sequences.npz                                   10,735,316.0
  ✓ attack_evaluation_sequences.npz                             3,954,752.0
  ✓ frozen_thresholds.json                                          1,347.0
  ✓ scaler.joblib                                                   2,127.0

[NB09] 11/11 present
  ✓ event_recall.csv                                                  456.0
  ✓ event_vs_point.csv                                              1,373.0
  ✓ detection_delays.csv                                           15,121.0
  ✓ detection_delay_summary.csv                                       410.0
  ✓ scenario_detection.csv                                          9,923.0
  ✓ scenario_detection_summary.csv            

In [4]:
# NB11 Cell 2 — Comparability audit (state)
# Documents frozen baseline state, preprocessing, score conventions, partitions.
# Writes NB11_STAGE_B_COMPARABILITY_STATE.md.

import json
import numpy as np
import pandas as pd

EXPECTED_METHODS = ["rule", "rolling", "simple", "if", "ocsvm", "lof", "lstm", "tcn", "usad"]

# Load frozen baseline
eval_summary = pd.read_csv(EVALUATION_SUMMARY).set_index("Method")
missing = [m for m in EXPECTED_METHODS if m not in eval_summary.index]
assert not missing, f"Baseline missing methods: {missing}"

print("=" * 72)
print("Frozen NB08 baseline — all 9 methods present")
print("=" * 72)
print(eval_summary[[
    "pAUC_05", "ROC_AUC", "PR_AUC", "F1_Frozen_Thr",
    "Precision", "Recall", "FPR_at_Thr",
]].round(4).to_string())

# Load frozen thresholds
frozen_thr = json.loads(FROZEN_THRESHOLDS.read_text())
print(f"\nFrozen thresholds loaded: {len(frozen_thr)} methods")

# Verify partition sizes from .npz shapes
seq = np.load(FINAL_SEQUENCE_SCORES)
obs = np.load(FINAL_OBSERVATION_SCORES)

partition_sizes = {
    "threshold_obs":   len(obs["rule_threshold_scores"]),
    "threshold_seq":   len(seq["tcn_threshold_seq_scores"]),
    "validation_obs":  len(obs["rule_validation_scores"]),
    "validation_seq":  len(seq["tcn_validation_seq_scores"]),
    "attack_obs":      len(obs["rule_attack_scores"]),
    "attack_seq":      len(seq["tcn_attack_seq_scores"]),
}
print("\n" + "-" * 72)
print("Partition sizes verified from .npz shapes")
print("-" * 72)
for k, v in partition_sizes.items():
    print(f"  {k:18s} {v:>8,}")

# Build markdown report
table_md = eval_summary[[
    "pAUC_05", "ROC_AUC", "PR_AUC", "F1_Frozen_Thr",
    "Precision", "Recall", "FPR_at_Thr",
]].round(4).to_markdown()

report_md = f"""# NB11 Stage B — Comparability Audit (State)

## 1. Baseline accessibility

- Frozen baseline loaded from `evaluation_summary.csv`
- All 9 expected methods present: {", ".join(sorted(eval_summary.index))}
- Frozen thresholds loaded from `frozen_thresholds.json` ({len(frozen_thr)} methods)

## 2. Frozen baseline metrics

{table_md}

## 3. Data partitions (verified from .npz shapes)

| Partition | Observation rows | Sequence rows (w30) |
|---|---:|---:|
| Threshold | {partition_sizes['threshold_obs']:,} | {partition_sizes['threshold_seq']:,} |
| Validation | {partition_sizes['validation_obs']:,} | {partition_sizes['validation_seq']:,} |
| Attack | {partition_sizes['attack_obs']:,} | {partition_sizes['attack_seq']:,} |

Observation counts match the documented NB08 protocol: 47,437 / 79,060 / 54,621.
Sequence counts at window 30 reflect the loss of the first 29 observations
per partition (no full window available): 47,408 / 79,031 / 54,592.

## 4. Preprocessing (frozen, shared by all learned methods)

- Scaler: MinMaxScaler, fit on the training partition only (`scaler.joblib`)
- Features: 42 sensor columns in NB08 order (`scaler.feature_names_in_`)
- Training rows: 221,366; monitoring rows: 47,435
- Chronological split preserved: training -> monitoring -> threshold -> validation
- Attack partition is a separate evaluation set

Baselines (rule, rolling, simple) use their own established feature definitions
from NB04/NB06. Feature-set differences are inherited from NB08 and documented
there; not re-derived in NB11.

## 5. Score orientation and threshold semantics

| Method class | Score type | Orientation | Threshold rule |
|---|---|---|---|
| Deep learning (LSTM, TCN, USAD) | Float reconstruction error | Higher = more anomalous (`score > thr`) | 99th pct of threshold partition |
| Traditional ML (IF, OCSVM, LOF) | Float decision function / distance | Higher = more anomalous (verified per-method in NB10) | 99th pct of threshold partition |
| Baselines (rule, rolling, simple) | Integer 0/1 decision | Higher = more anomalous (`score > 0`) | Fixed, established in NB04/NB06 |

Score orientation for score-based methods was verified in NB10's threshold
cell by matching frozen TP/FP counts per method. It is not re-verified here.

## 6. Sequence endpoint convention (the alignment issue)

Sequence-based methods score at the last observation of each window. For a
partition of N observation rows and window W, sequence count is N - W + 1:

- Window 30: first 29 observations per partition cannot receive a score
- Window 60: first 59 observations per partition cannot receive a score
- Window 90: first 89 observations per partition cannot receive a score

This is why w30, w60, w90 have different evaluated observation counts. Cell 3
quantifies how much this matters for the window-sensitivity comparison.

## 7. Known limitations inherited from NB08

Documented in NB08 / NB09 and carried forward unchanged:

- USAD: best_epoch = 1, adversarial term inactive. Weak detector, documented
  as such in NB08. Not retrained in NB11.
- LOF: ROC-AUC 0.19 — score direction anti-correlated with labels as
  configured. No threshold can repair this. Documented in NB08.
- Rule-Based: FPR 92.74% at fixed threshold. No continuous score — quantile
  thresholds are not applicable (reported fixed-only in NB10).
- Feature sets: baselines and learned methods use different feature
  definitions. Documented in NB04/NB06. Not harmonised in NB11.

## 8. Baseline reproducibility

Verified in NB10, cited here (not re-run):

- 18/18 pAUC@0.05 and ROC-AUC checks match `evaluation_summary.csv` to 1e-4
- 9/9 p99 integer-count reproductions (TP and FP match frozen exactly)
- 3/3 baseline fixed-threshold reproductions

The frozen baseline is intact and reproducible. NB11 does not re-verify it.

## 9. What Cell 3 verifies

Cell 3 will:
- Re-score the 4 window models (w60/w90 for TCN and LSTM) since `window_scores.npz`
  is missing from the local NB10 extraction
- Align w30/w60/w90 scores to a common observation set per partition
- Recompute pAUC / F1 / ROC-AUC on the aligned set
- Report whether NB10's window deltas survive alignment

## 10. What carries forward to Cell 4

The unified comparison table (Cell 4) will join:
- Point-level metrics from this frozen baseline
- Event-level metrics from NB09
- Robustness findings from NB10
- Computational cost from NB10 pilot metadata

No method, metric, or configuration is re-derived. Every row traces to a
specific artifact in NB08, NB09, or NB10.
"""

report_path = NB11_OUT_DIR / "NB11_STAGE_B_COMPARABILITY_STATE.md"
report_path.write_text(report_md, encoding="utf-8")

print("\n" + "=" * 72)
print("Stage B (state) complete")
print("=" * 72)
print("  Baseline: 9/9 methods, all metrics present")
print("  Partitions: verified against .npz shapes")
print(f"  Frozen thresholds: {len(frozen_thr)} loaded")
print("  Baseline reproducibility: cited from NB10, not re-run")
print(f"  Report: {report_path.name}")
print("  Next: Cell 3 (alignment audit)")
print("=" * 72)

Frozen NB08 baseline — all 9 methods present
         pAUC_05  ROC_AUC  PR_AUC  F1_Frozen_Thr  Precision  Recall  FPR_at_Thr
Method                                                                         
tcn       0.7597   0.8240  0.8778         0.8629     0.9449  0.7940      0.0320
lstm      0.7289   0.8513  0.8925         0.8386     0.9478  0.7519      0.0286
usad      0.6438   0.7934  0.8471         0.7723     0.9654  0.6436      0.0160
simple    0.6434   0.8839  0.9056         0.8144     0.8059  0.8231      0.1369
ocsvm     0.6410   0.7802  0.8294         0.7693     0.9815  0.6325      0.0082
if        0.6266   0.7754  0.8294         0.7470     0.9664  0.6088      0.0146
rule      0.6187   0.7686  0.8207         0.5417     0.3933  0.8703      0.9274
lof       0.0584   0.1946  0.3276         0.1117     0.6098  0.0615      0.0272
rolling   0.0161   0.4711  0.3779         0.3727     0.3796  0.3661      0.4134

Frozen thresholds loaded: 10 methods

------------------------------------

In [5]:
# NB11 Cell 3a — Ensure window_scores.npz exists

import gc
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

RECOMPUTE_NEEDED = not (WINDOW_SCORES and WINDOW_SCORES.is_file()
                        and WINDOW_SCORES_META and WINDOW_SCORES_META.is_file())

print("=" * 72)
print("Cell 3a — window_scores.npz")
print("=" * 72)

if not RECOMPUTE_NEEDED:
    print(f"Already present: {WINDOW_SCORES.name} ({WINDOW_SCORES.stat().st_size:,} bytes)")
    print(f"Already present: {WINDOW_SCORES_META.name}")
    print("\nSkipping re-score. Proceed to Cell 3b.")
else:
    print("window_scores.npz missing. Will re-score the 4 window models.")
    print("PyTorch required.")

    # Check dependencies before doing any work
    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
    except ImportError as exc:
        print(f"\nPyTorch not available: {exc}")
        print("Install with: pip install torch")
        print("Or run Cell 3a on Kaggle where PyTorch is pre-installed.")
        raise SystemExit("Cannot re-score without PyTorch.")

    DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    print(f"Device: {DEVICE}")

    # Sanity: check inputs
    PILOT_DIR = NB10_DIR / "pilot"
    needed = [
        PILOT_DIR / "tcn_window60_seed42_minmax.pth",
        PILOT_DIR / "tcn_window90_seed42_minmax.pth",
        PILOT_DIR / "lstm_window60_seed42_minmax.pth",
        PILOT_DIR / "lstm_window90_seed42_minmax.pth",
        NB08_DIR / "scaler.joblib",
        NORMAL_CSV,
        ATTACK_CSV,
    ]
    missing = [p for p in needed if not p.is_file()]
    if missing:
        print("\nMissing inputs for re-scoring:")
        for p in missing:
            print(f"  {p.name}")
        raise FileNotFoundError("Re-scoring inputs incomplete.")

    print("\nInputs present:")
    for p in needed:
        print(f"  {p.name}")

    # Constants (mirror NB10 Phase 5d)
    TRAIN_ROWS = 221_366
    MONITOR_ROWS = 47_435
    THRESHOLD_ROWS = 47_437
    VALIDATION_ROWS = 79_060
    TIMESTAMP_COL = "Timestamp"
    CHUNK_SIZE = 100_000
    N_FEATURES = 42
    THRESHOLD_QUANTILE = 0.99

    MODELS = [
        ("tcn_window60",  "tcn",  60, PILOT_DIR / "tcn_window60_seed42_minmax.pth"),
        ("tcn_window90",  "tcn",  90, PILOT_DIR / "tcn_window90_seed42_minmax.pth"),
        ("lstm_window60", "lstm", 60, PILOT_DIR / "lstm_window60_seed42_minmax.pth"),
        ("lstm_window90", "lstm", 90, PILOT_DIR / "lstm_window90_seed42_minmax.pth"),
    ]

    # Load scaler
    scaler = joblib.load(NB08_DIR / "scaler.joblib")
    FEATURES = list(scaler.feature_names_in_)
    assert len(FEATURES) == N_FEATURES, f"Expected {N_FEATURES} features, got {len(FEATURES)}"

    # Load data partitions (same procedure as NB10 Phase 5d)
    def load_normal_partitions():
        header = [str(c).strip() for c in pd.read_csv(NORMAL_CSV, nrows=0).columns]
        LABEL_COL = next((c for c in header if c.lower() in {"normal/attack", "label"}), None)
        SENSOR_COLS = [c for c in header if c not in {TIMESTAMP_COL, LABEL_COL}]
        rows_needed = TRAIN_ROWS + MONITOR_ROWS + THRESHOLD_ROWS + VALIDATION_ROWS
        parts, collected = [], 0
        for chunk in pd.read_csv(NORMAL_CSV, chunksize=CHUNK_SIZE, low_memory=False):
            chunk.columns = [str(c).strip() for c in chunk.columns]
            chunk[TIMESTAMP_COL] = pd.to_datetime(chunk[TIMESTAMP_COL], errors="coerce")
            keep = chunk[TIMESTAMP_COL].notna() & chunk[SENSOR_COLS].notna().all(axis=1)
            eligible = chunk.loc[keep]
            if eligible.empty:
                continue
            take = min(len(eligible), rows_needed - collected)
            parts.append(eligible[[TIMESTAMP_COL] + SENSOR_COLS].iloc[:take].copy())
            collected += take
            if collected >= rows_needed:
                break
        df = pd.concat(parts, ignore_index=True)
        thr_start = TRAIN_ROWS + MONITOR_ROWS
        val_start = thr_start + THRESHOLD_ROWS
        return (
            df.iloc[thr_start:val_start][FEATURES].astype(np.float32).to_numpy(),
            df.iloc[val_start:][FEATURES].astype(np.float32).to_numpy(),
        )

    def load_attack_partition():
        df = pd.read_csv(ATTACK_CSV, low_memory=False)
        df.columns = df.columns.str.strip()
        df[TIMESTAMP_COL] = pd.to_datetime(
            df[TIMESTAMP_COL].astype("string").str.strip(),
            format="%d/%m/%Y %I:%M:%S %p", errors="raise",
        )
        return df[FEATURES].astype(np.float32).to_numpy()

    print("\nLoading data partitions...")
    thr_raw, val_raw = load_normal_partitions()
    atk_raw = load_attack_partition()
    thr_s = scaler.transform(thr_raw).astype(np.float32)
    val_s = scaler.transform(val_raw).astype(np.float32)
    atk_s = scaler.transform(atk_raw).astype(np.float32)
    del thr_raw, val_raw, atk_raw
    gc.collect()
    print(f"  threshold: {thr_s.shape}")
    print(f"  validation: {val_s.shape}")
    print(f"  attack: {atk_s.shape}")

    # Sequence construction and scoring
    def create_sequences(data, window_size):
        w = np.lib.stride_tricks.sliding_window_view(data, window_shape=window_size, axis=0)
        w = np.moveaxis(w, -1, 1)
        return np.ascontiguousarray(w, dtype=np.float32)

    def score_sequences(model, seqs, batch=512):
        model.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(seqs), batch):
                b = torch.from_numpy(seqs[i:i + batch]).to(DEVICE)
                r = model(b)
                out.append(((r - b) ** 2).mean(dim=(1, 2)).cpu().numpy())
        return np.concatenate(out).astype(np.float64)

    # Architectures (state-dict compatible with NB10 checkpoints)
    class LSTMAutoencoder(nn.Module):
        def __init__(self, input_size, hidden_size, num_layers, bottleneck_size, dropout):
            super().__init__()
            self.encoder = nn.LSTM(input_size, hidden_size, num_layers,
                                   batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.encoder_projection = nn.Linear(hidden_size, bottleneck_size)
            self.decoder_projection = nn.Linear(bottleneck_size, hidden_size)
            self.decoder = nn.LSTM(hidden_size, hidden_size, num_layers,
                                   batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.output_layer = nn.Linear(hidden_size, input_size)

        def forward(self, x):
            _, (h, _) = self.encoder(x)
            lat = F.leaky_relu(self.encoder_projection(h[-1]), negative_slope=0.01)
            di = F.leaky_relu(self.decoder_projection(lat), negative_slope=0.01)
            di = di.unsqueeze(1).repeat(1, x.size(1), 1)
            d, _ = self.decoder(di)
            return self.output_layer(d)

    class CausalConv1d(nn.Module):
        def __init__(self, in_c, out_c, k, d):
            super().__init__()
            self.left_padding = (k - 1) * d
            self.conv = nn.Conv1d(in_c, out_c, kernel_size=k, dilation=d, padding=0)

        def forward(self, x):
            return self.conv(F.pad(x, (self.left_padding, 0)))

    class TCNBlock(nn.Module):
        def __init__(self, in_c, out_c, k, d):
            super().__init__()
            self.conv1 = CausalConv1d(in_c, out_c, k, d)
            self.conv2 = CausalConv1d(out_c, out_c, k, d)
            self.activation = nn.ReLU()
            self.residual = (nn.Identity() if in_c == out_c
                             else nn.Conv1d(in_c, out_c, kernel_size=1))

        def forward(self, x):
            r = self.residual(x)
            x = self.activation(self.conv1(x))
            x = self.activation(self.conv2(x))
            return self.activation(x + r)

    class TCNAutoencoder(nn.Module):
        def __init__(self, n_features, channels, kernel_size, dilations):
            super().__init__()
            enc, in_c = [], n_features
            for oc, d in zip(channels, dilations):
                enc.append(TCNBlock(in_c, oc, kernel_size, d))
                in_c = oc
            self.encoder = nn.Sequential(*enc)
            dec, rc, rd = [], list(reversed(channels)), list(reversed(dilations))
            for i, d in enumerate(rd):
                cur = rc[i]
                nxt = rc[i + 1] if i + 1 < len(rc) else n_features
                dec.append(CausalConv1d(cur, nxt, kernel_size, d))
                if i + 1 < len(rd):
                    dec.append(nn.ReLU())
            self.decoder = nn.Sequential(*dec)

        def forward(self, x):
            x = x.transpose(1, 2).contiguous()
            x = self.decoder(self.encoder(x))
            return x.transpose(1, 2).contiguous()

    def build_tcn():
        return TCNAutoencoder(N_FEATURES, [32, 32, 32, 32], 3, [1, 2, 4, 8]).to(DEVICE)

    def build_lstm():
        return LSTMAutoencoder(N_FEATURES, 64, 2, 16, 0.2).to(DEVICE)

    # Score each model
    saved_arrays = {}
    metadata = {
        "partitions": {
            "threshold_rows": THRESHOLD_ROWS,
            "validation_rows": VALIDATION_ROWS,
            "attack_rows": 54_621,
        },
        "models": {},
    }

    print("\n" + "-" * 72)
    print("Re-scoring 4 window models")
    print("-" * 72)

    for name, arch, window, ckpt in MODELS:
        print(f"\n{name}  window={window}  ckpt={ckpt.name}")

        thr_seq = create_sequences(thr_s, window)
        val_seq = create_sequences(val_s, window)
        atk_seq = create_sequences(atk_s, window)
        print(f"  sequences: thr={len(thr_seq):,}  val={len(val_seq):,}  atk={len(atk_seq):,}")

        model = build_tcn() if arch == "tcn" else build_lstm()
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        model.eval()

        t0 = time.perf_counter()
        thr_scores = score_sequences(model, thr_seq)
        val_scores = score_sequences(model, val_seq)
        atk_scores = score_sequences(model, atk_seq)
        elapsed = time.perf_counter() - t0

        threshold = float(np.percentile(thr_scores, THRESHOLD_QUANTILE * 100, method="linear"))

        saved_arrays[f"{name}_thr_scores"] = thr_scores
        saved_arrays[f"{name}_val_scores"] = val_scores
        saved_arrays[f"{name}_atk_scores"] = atk_scores
        saved_arrays[f"{name}_thr_end_indices"] = np.arange(window - 1, window - 1 + len(thr_scores))
        saved_arrays[f"{name}_val_end_indices"] = np.arange(window - 1, window - 1 + len(val_scores))
        saved_arrays[f"{name}_atk_end_indices"] = np.arange(window - 1, window - 1 + len(atk_scores))

        per_1k = elapsed / ((len(thr_scores) + len(val_scores) + len(atk_scores)) / 1000.0)
        metadata["models"][name] = {
            "arch": arch,
            "window": window,
            "checkpoint": ckpt.name,
            "threshold": threshold,
            "n_thr": int(len(thr_scores)),
            "n_val": int(len(val_scores)),
            "n_atk": int(len(atk_scores)),
            "inference_seconds_total": float(elapsed),
            "inference_seconds_per_1k_seqs": float(per_1k),
        }
        print(f"  threshold={threshold:.6g}  inference={elapsed:.2f}s  per-1k={per_1k:.2f}s")

        del model, thr_seq, val_seq, atk_seq, thr_scores, val_scores, atk_scores
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    # Save
    WS_DIR = NB10_DIR / "window_scores"
    WS_DIR.mkdir(parents=True, exist_ok=True)
    ws_npz = WS_DIR / "window_scores.npz"
    ws_meta = WS_DIR / "window_scores_metadata.json"

    print(f"\nSaving {ws_npz.name}...")
    np.savez_compressed(ws_npz, **saved_arrays)
    ws_meta.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"  npz size: {ws_npz.stat().st_size:,} bytes")
    print(f"  metadata: {ws_meta.name}")

    print("\n" + "-" * 72)
    print("Summary")
    print("-" * 72)
    for name, m in metadata["models"].items():
        print(f"  {name:14s} thr={m['threshold']:.6g}  "
              f"n_thr={m['n_thr']:,}  n_val={m['n_val']:,}  n_atk={m['n_atk']:,}  "
              f"inference={m['inference_seconds_total']:.2f}s")

print("\n" + "=" * 72)
print("Cell 3a complete. Proceed to Cell 3b (alignment audit).")
print("=" * 72)

Cell 3a — window_scores.npz
window_scores.npz missing. Will re-score the 4 window models.
PyTorch required.
Device: cpu

Inputs present:
  tcn_window60_seed42_minmax.pth
  tcn_window90_seed42_minmax.pth
  lstm_window60_seed42_minmax.pth
  lstm_window90_seed42_minmax.pth
  scaler.joblib
  normal.csv
  attack.csv


c:\Users\ADEGOKE\Desktop\SWATS\cats\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Loading data partitions...


C:\Users\ADEGOKE\AppData\Local\Temp\ipykernel_5592\3938459512.py:94: UserWarning: Parsing dates in  %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  chunk[TIMESTAMP_COL] = pd.to_datetime(chunk[TIMESTAMP_COL], errors="coerce")


KeyboardInterrupt: 